# MGMT298D: Science and Strategy of AI
### Week 8B - AI Agents
### Application: Autonomous Task Completion

## Setup

In [ ]:
!pip install -q -U google-generativeai pandas

import pandas as pd
import google.generativeai as genai
from google.colab import userdata
import json
import time

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-1.5-flash-latest')

def generate(prompt, temperature=0.7):
    response = model.generate_content(prompt, generation_config={'temperature': temperature})
    return response.text

print("Setup complete")

## What is an AI Agent?

An **AI agent** is an LLM that can:
1. **Plan** - Break down a goal into steps
2. **Use tools** - Execute code, search the web, query databases
3. **Observe** - Process the results of actions
4. **Iterate** - Adjust plans based on observations

Unlike a simple chatbot, an agent takes autonomous action to achieve a goal.

In [ ]:
# Simple illustration: Chatbot vs Agent
comparison = """
╔══════════════════════════════════════════════════════════════════╗
║                    CHATBOT vs AI AGENT                           ║
╠══════════════════════════════════════════════════════════════════╣
║  CHATBOT (Reactive)              │  AI AGENT (Proactive)         ║
║  ─────────────────────           │  ─────────────────────        ║
║  User: "What's the weather?"     │  User: "Plan my trip to NYC" ║
║  Bot: "I don't have access       │  Agent:                       ║
║        to weather data."         │    1. Search for flights      ║
║                                   │    2. Check weather forecast  ║
║  (Single response, no action)    │    3. Find hotels near events ║
║                                   │    4. Create itinerary        ║
║                                   │    5. Book reservations       ║
║                                   │  (Multi-step, takes actions)  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(comparison)

## The Agent Loop: Plan → Act → Observe → Repeat

In [ ]:
# Define simple "tools" the agent can use
def search_database(query):
    """Simulated database search."""
    data = {
        "revenue Q1": "$2.3M",
        "revenue Q2": "$2.8M",
        "top product": "Widget Pro ($890K)",
        "customer count": "1,247",
        "churn rate": "4.2%"
    }
    for key, value in data.items():
        if query.lower() in key.lower():
            return f"Found: {key} = {value}"
    return "No matching data found."

def calculate(expression):
    """Simple calculator tool."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except:
        return "Calculation error"

def send_email(to, subject, body):
    """Simulated email sending."""
    return f"Email sent to {to} with subject: '{subject}'"

TOOLS = {
    "search_database": search_database,
    "calculate": calculate,
    "send_email": send_email
}

print("Available tools:", list(TOOLS.keys()))

In [ ]:
def run_agent(goal, max_steps=5):
    """Simple agent that plans and executes steps to achieve a goal."""
    
    print(f"\n{'='*60}")
    print(f"GOAL: {goal}")
    print(f"{'='*60}\n")
    
    context = f"Goal: {goal}\n\nAvailable tools: search_database(query), calculate(expression), send_email(to, subject, body)\n"
    history = []
    
    for step in range(max_steps):
        # PLAN: Ask LLM what to do next
        plan_prompt = f"""
{context}

Previous actions and results:
{chr(10).join(history) if history else 'None yet'}

What is the SINGLE next action to take? Respond in this JSON format:
{{
  "thought": "your reasoning",
  "action": "tool_name",
  "action_input": "input for the tool",
  "is_complete": true/false
}}

If the goal is complete, set is_complete to true and action to "finish".
"""
        
        response = generate(plan_prompt, temperature=0.3)
        
        # Parse the response
        try:
            clean = response.replace('```json', '').replace('```', '').strip()
            action_data = json.loads(clean)
        except:
            print(f"Step {step+1}: Failed to parse response")
            continue
        
        print(f"Step {step+1}:")
        print(f"  Thought: {action_data.get('thought', 'N/A')}")
        print(f"  Action: {action_data.get('action', 'N/A')}")
        
        # Check if complete
        if action_data.get('is_complete', False):
            print(f"\n✓ Goal completed!")
            return history
        
        # ACT: Execute the tool
        action = action_data.get('action', '')
        action_input = action_data.get('action_input', '')
        
        if action in TOOLS:
            # Handle different input formats
            if action == 'send_email' and isinstance(action_input, dict):
                result = TOOLS[action](**action_input)
            else:
                result = TOOLS[action](action_input)
            print(f"  Result: {result}")
        else:
            result = f"Unknown tool: {action}"
            print(f"  Result: {result}")
        
        # OBSERVE: Record the result
        history.append(f"Action: {action}({action_input}) → {result}")
        print()
    
    print("Max steps reached")
    return history

In [ ]:
# Run the agent with a business goal
history = run_agent("Find Q1 and Q2 revenue, calculate the growth rate, and email the results to manager@company.com")

## Real-World Agent Capabilities

Production AI agents can use many more tools:

In [ ]:
agent_capabilities = """
╔════════════════════════════════════════════════════════════════════╗
║                    AI AGENT CAPABILITIES                            ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  DATA & ANALYSIS                │  COMMUNICATION                   ║
║  ────────────────               │  ─────────────                   ║
║  • Query SQL databases          │  • Send emails                   ║
║  • Read/write spreadsheets      │  • Post to Slack                 ║
║  • Analyze CSV files            │  • Create documents              ║
║  • Generate charts              │  • Schedule meetings             ║
║                                                                     ║
║  WEB & SEARCH                   │  CODE & AUTOMATION               ║
║  ───────────                    │  ─────────────────               ║
║  • Browse websites              │  • Write & run Python            ║
║  • Search Google                │  • Execute shell commands        ║
║  • Call external APIs           │  • Manage files                  ║
║  • Scrape data                  │  • Deploy applications           ║
║                                                                     ║
╠════════════════════════════════════════════════════════════════════╣
║  EXAMPLES OF AGENT PRODUCTS:                                        ║
║  • GitHub Copilot Workspace - Plans and implements code changes     ║
║  • Devin - Autonomous software engineer                            ║
║  • AutoGPT - General-purpose task automation                       ║
║  • Claude Computer Use - Controls desktop applications              ║
╚════════════════════════════════════════════════════════════════════╝
"""
print(agent_capabilities)

## Agent with Data Analysis

In [ ]:
# Create sample sales data
import numpy as np

np.random.seed(42)
sales_data = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C', 'Widget D'] * 25,
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100),
    'revenue': np.random.randint(1000, 10000, 100),
    'units': np.random.randint(10, 100, 100)
})

print("Sample sales data:")
print(sales_data.head(10))

In [ ]:
def data_agent(question, df):
    """Agent that analyzes data to answer business questions."""
    
    # Step 1: Plan the analysis
    plan_prompt = f"""
You are a data analyst. Given this DataFrame with columns: {list(df.columns)}
Sample data:
{df.head(3).to_string()}

Question: {question}

Write Python code using pandas to answer this question.
The DataFrame is named 'df'. Return ONLY the code, no explanation.
"""
    
    code = generate(plan_prompt, temperature=0.2)
    code = code.replace('```python', '').replace('```', '').strip()
    
    print("Generated code:")
    print(code)
    print()
    
    # Step 2: Execute the code
    try:
        result = eval(code)
        print("Result:")
        print(result)
        return result
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# Ask the data agent business questions
print("=" * 60)
print("Question: What is the total revenue by region?")
print("=" * 60)
data_agent("What is the total revenue by region?", sales_data)

In [ ]:
print("=" * 60)
print("Question: Which product has the highest average revenue per unit?")
print("=" * 60)
data_agent("Which product has the highest average revenue per unit?", sales_data)

## Agent Architectures

In [ ]:
architectures = """
╔════════════════════════════════════════════════════════════════════╗
║                    COMMON AGENT ARCHITECTURES                       ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  1. ReAct (Reasoning + Acting)                                      ║
║     ┌──────────┐     ┌──────────┐     ┌──────────┐                 ║
║     │  Think   │ ──► │   Act    │ ──► │ Observe  │ ──► (repeat)   ║
║     └──────────┘     └──────────┘     └──────────┘                 ║
║     "I should search..." → search() → "Found X..." → "Now I..."  ║
║                                                                     ║
║  2. Plan-and-Execute                                                ║
║     ┌──────────┐     ┌──────────┐     ┌──────────┐                 ║
║     │  Plan    │ ──► │ Execute  │ ──► │ Verify   │                 ║
║     │  All     │     │   All    │     │  Result  │                 ║
║     └──────────┘     └──────────┘     └──────────┘                 ║
║     Create full plan upfront, then execute steps sequentially       ║
║                                                                     ║
║  3. Multi-Agent                                                     ║
║     ┌──────────┐                                                   ║
║     │ Manager  │                                                   ║
║     └────┬─────┘                                                   ║
║          │                                                          ║
║     ┌────┴────┬────────┬────────┐                                  ║
║     ▼         ▼        ▼        ▼                                  ║
║  ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐                              ║
║  │Coder │ │Tester│ │Writer│ │Review│                              ║
║  └──────┘ └──────┘ └──────┘ └──────┘                              ║
║     Specialized agents collaborate on complex tasks                 ║
║                                                                     ║
╚════════════════════════════════════════════════════════════════════╝
"""
print(architectures)

## Challenges and Limitations

In [ ]:
challenges = """
╔════════════════════════════════════════════════════════════════════╗
║                    AGENT CHALLENGES                                 ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  RELIABILITY                    │  SAFETY                          ║
║  ───────────                    │  ──────                          ║
║  • Agents can get stuck         │  • May take unintended actions   ║
║  • Plans may be incorrect       │  • Access to sensitive tools     ║
║  • Error propagation            │  • Need guardrails & limits      ║
║  • Inconsistent outputs         │  • Human approval for critical   ║
║                                                                     ║
║  COST                           │  EVALUATION                      ║
║  ────                           │  ──────────                      ║
║  • Many API calls per task      │  • Hard to test all scenarios   ║
║  • Long context = high cost     │  • Success metrics unclear       ║
║  • Trial-and-error expensive    │  • Edge cases are common         ║
║                                                                     ║
╠════════════════════════════════════════════════════════════════════╣
║  BEST PRACTICES:                                                    ║
║  • Start with narrow, well-defined tasks                           ║
║  • Add human-in-the-loop for important decisions                   ║
║  • Set clear boundaries on what tools can do                        ║
║  • Log all actions for debugging and audit                          ║
║  • Test extensively before production                               ║
╚════════════════════════════════════════════════════════════════════╝
"""
print(challenges)